# Construction-Site PPE Safety Detection
## Reproducibility and Experimental Validation Notebook

**MAICEN-0526 - Group 5**

| Team Member | Role |
|---|---|
| Dru | Group Lead |
| Shaheen | Group Member |
| Ghandoor | Group Member |
| Ahamed | Group Member |

---

### Notebook Purpose

- Reproduce and document the key technical workflow used for the construction-site PPE object-detection project.
- Provide a transparent record of the experimental methodology used for the M1 baseline and M2 targeted-augmentation models.
- Demonstrate the environment, dependencies, configurations, evaluation logic and supporting analysis required for reproducibility.
- Complement the Roboflow experimental evidence and the project GitHub repository.
- Document limitations and responsible-use considerations relevant to safety-related computer vision.

### Experimental Workflow

**M1 Baseline → External Challenge Testing → Error Analysis → Targeted Augmentation → M2 Refined Model → Controlled Comparison → Reproducibility → Governance**

---

## 1. Executive Summary

- **Problem:** Construction-site PPE monitoring requires reliable identification of workers and safety-related PPE conditions from images.

- **Approach:** A YOLO11 Nano object-detection model was evaluated through a controlled experimental workflow consisting of a baseline model (M1), external challenge testing, structured error analysis and a targeted-augmentation model (M2).

- **Dataset:** 3,858 annotated source images across seven classes: `Person`, `Helmet`, `Non-Helmet`, `Vest`, `Gloves`, `Shoes` and `bare-arms`.

- **Baseline performance:** M1 achieved **96.9% mAP@50**, **97.1% precision**, **94.2% recall** and **95.6% F1**.

- **Experimental refinement:** M2 introduced horizontal flipping, 0-10% random cropping, ±15° rotation and ±15% brightness variation while retaining the same validation and test sets.

- **M2 performance:** M2 achieved **96.7% mAP@50**, **95.7% precision** and **94.8% recall**. The weakest M1 test class, `bare-arms`, improved from **91% to 93% mAP@50**.

- **Key finding:** Strong held-out performance did not eliminate generalisation weaknesses observed during external M1 challenge testing. The experiment therefore demonstrates the importance of error analysis, class-level evaluation and external validation rather than relying only on headline mAP.

- **Reproducibility goal:** This notebook documents the experimental configuration, environment, evaluation logic and supporting results required to understand and reproduce the project workflow.

- **Responsible-use position:** The model is an experimental safety-monitoring aid and is not intended to replace human judgement or act as an authoritative determination of PPE compliance.

### Expected Outcome

- A clear and reproducible technical record of the Group 5 experiment.
- Transparent comparison of M1 and M2 rather than selectively reporting only favourable results.
- Explicit documentation of model limitations and safety-related interpretation constraints.

---


## 2. Table of Contents

1. **Executive Summary**
2. **Table of Contents**
3. **Experimental Objectives and Reproducibility Scope**
4. **Computational Environment and Dependencies**
5. **Dataset and Class Ontology**
6. **M1 - Baseline Experimental Configuration**
7. **M1 - Performance Results**
8. **External Challenge Testing and Error Analysis**
9. **M2 - Targeted Augmentation Strategy**
10. **M2 - Performance Results**
11. **Controlled M1 vs M2 Comparison**
12. **Reproducibility Verification**
13. **Limitations and Responsible Use**
14. **Conclusions and Recommended Future Work**
15. **References and Supporting Resources**

---

### Notebook Navigation

- Sections follow the experimental workflow in chronological order.
- Markdown cells document the **purpose, rationale and expected outcome** of each technical stage.
- Code cells provide the corresponding **reproducible implementation or verification step** where appropriate.
- Outputs are retained as experimental evidence wherever they materially support the analysis.

---

## 3. Experimental Objectives and Reproducibility Scope

### 3.1 Experimental Objectives

- Establish a **baseline PPE object-detection model (M1)** using YOLO11 Nano and the unaugmented dataset version.
- Evaluate M1 using both **held-out validation/test data** and **external challenge images**.
- Identify meaningful failure modes through structured **error analysis** rather than relying solely on aggregate performance metrics.
- Develop a **targeted augmentation strategy** based on the observed external failure modes.
- Train a refined model **(M2)** while preserving the original validation and test sets for controlled comparison.
- Compare M1 and M2 using both headline and class-level performance metrics.
- Document limitations relevant to the responsible use of object detection in a construction-safety context.

### 3.2 Reproducibility Scope

- Record the software environment and principal Python dependencies.
- Document the dataset versions, preprocessing, augmentation and model configurations used in the experiments.
- Preserve the numerical M1 and M2 evaluation results in a structured and independently inspectable form.
- Reproduce the principal comparative analysis and visualisations from the recorded experimental results.
- Maintain fixed evaluation assumptions where applicable, including the **50% confidence** and **50% overlap** thresholds used for M1 external challenge testing.
- Clearly distinguish between results reproduced computationally in this notebook and results originally generated through Roboflow hosted training and evaluation.

### 3.3 Reproducibility Boundary

- This notebook does **not** claim to retrain the hosted M1 and M2 models from scratch.
- Original model training was performed through **Roboflow hosted training** and is documented through the preserved experimental configuration and results.
- The notebook focuses on making the **methodology, configuration, results, comparison and analytical workflow reproducible and auditable** without unnecessarily consuming additional hosted-training resources.
- External M1 challenge tests are documented as **qualitative generalisation tests**, not as a formally annotated external mAP benchmark.
- No unsupported external M2 performance claim is made because equivalent hosted external inference evidence was not available during the experimental period.

### Expected Outcome

- An evaluator can identify exactly **what was reproduced, what was originally generated in Roboflow, and what evidence supports each stage**.
- The notebook avoids presenting previously recorded results as though they were newly generated by notebook code.
- Experimental constraints are documented transparently rather than hidden or treated as model failures.

---

## 4. Computational Environment and Dependencies

### 4.1 Purpose

- Record the computational environment used to execute this reproducibility notebook.
- Identify the Python version and principal software dependencies available during execution.
- Provide environment information that can assist another user in reproducing or diagnosing the notebook workflow.

### 4.2 Rationale

- Reproducibility depends not only on code, but also on the software environment in which that code is executed.
- Library versions may affect model loading, image processing, numerical calculations and visualisation behaviour.
- Recording package versions provides an auditable snapshot of the execution environment.
- The repository also contains a `requirements.txt` file documenting the principal project dependencies.

### Expected Outcome

- Confirm that the Colab Python environment is operational.
- Record the Python version and platform information.
- Verify availability of the principal libraries required by the project.
- Produce a clear environment record that can be retained with the notebook as reproducibility evidence.

---

In [10]:
# 4.3 Computational Environment Verification

import sys
import platform
import importlib.metadata as metadata

print("=" * 70)
print("MAICEN-0526 GROUP 5 - COMPUTATIONAL ENVIRONMENT")
print("=" * 70)

print(f"Python version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")

print("\nPrincipal package versions:")
print("-" * 70)

packages = [
    "ultralytics",
    "roboflow",
    "opencv-python",
    "numpy",
    "pandas",
    "matplotlib",
    "Pillow",
    "PyYAML"
]

for package in packages:
    try:
        version = metadata.version(package)
        print(f"{package:<20} {version}")
    except metadata.PackageNotFoundError:
        print(f"{package:<20} NOT INSTALLED")

print("=" * 70)
print("Environment verification complete.")

MAICEN-0526 GROUP 5 - COMPUTATIONAL ENVIRONMENT
Python version : 3.13.15
Platform       : Linux-6.6.122+-x86_64-with-glibc2.39

Principal package versions:
----------------------------------------------------------------------
ultralytics          8.4.156
roboflow             1.5.0
opencv-python        5.0.0.93
numpy                2.1.3
pandas               2.2.3
matplotlib           3.10.0
Pillow               11.3.0
PyYAML               6.0.3
Environment verification complete.


### 4.4 Environment Verification Result

- The Colab runtime was successfully initialised and the environment inspection completed without execution errors.
- The runtime is using **Python 3.13.15** on a **Linux x86_64** platform.
- Core analytical and image-processing dependencies including **OpenCV, NumPy, pandas, Matplotlib, Pillow and PyYAML** were already available.
- **Ultralytics** and **Roboflow** were not installed in the initial runtime.
- This initial inspection was intentionally performed **before installing additional packages**, providing a transparent record of the starting computational environment.
- Missing project-specific packages at this stage represent the state of the fresh runtime rather than a failure of the notebook.

### Expected Next Step

- Install the project-specific dependencies required for subsequent reproducibility checks.
- Re-run package verification after installation.
- Preserve both the **pre-installation** and **post-installation** outputs to demonstrate how the execution environment was prepared.

---


### 4.5 Project-Specific Dependency Installation

#### Purpose

- Install the principal project-specific Python packages that were not available in the initial Colab runtime.
- Prepare the environment for subsequent reproducibility and verification activities.
- Maintain a transparent distinction between packages supplied by the original Colab environment and packages installed specifically for this project.

#### Rationale

- The initial environment inspection confirmed that **Ultralytics** and **Roboflow** were not installed.
- **Ultralytics** provides the YOLO framework associated with the object-detection methodology used in this project.
- **Roboflow** provides supporting functionality associated with the dataset-management and hosted experimental workflow.
- Installing these dependencies explicitly makes the notebook environment easier to reproduce and audit.
- This installation step does **not** retrain M1 or M2 and does not consume Roboflow hosted-training credits.

#### Expected Outcome

- Ultralytics and Roboflow are successfully installed in the Colab runtime.
- Their installed versions can subsequently be recorded and verified.
- The environment is prepared for the remaining reproducibility workflow.

---

In [11]:
# ============================================================
# MAICEN-0526 GROUP 5 - PROJECT-SPECIFIC DEPENDENCIES
# ============================================================

%pip install -q ultralytics roboflow

print("=" * 70)
print("Project-specific dependency installation complete.")
print("=" * 70)

Project-specific dependency installation complete.


In [12]:
# ============================================================
# MAICEN-0526 GROUP 5 - POST-INSTALLATION VERIFICATION
# ============================================================

from importlib.metadata import version, PackageNotFoundError

packages = ["ultralytics", "roboflow"]

print("=" * 70)
print("MAICEN-0526 GROUP 5 - POST-INSTALLATION VERIFICATION")
print("=" * 70)

for package in packages:
    try:
        print(f"{package:<15}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:<15}: NOT INSTALLED")

print("=" * 70)
print("Post-installation verification complete.")

MAICEN-0526 GROUP 5 - POST-INSTALLATION VERIFICATION
ultralytics    : 8.4.156
roboflow       : 1.5.0
Post-installation verification complete.


### 4.6 Dependency Installation and Verification Result

- The project-specific dependency installation completed successfully.
- Post-installation verification confirmed **Ultralytics 8.4.156** and **Roboflow 1.5.0** in the active Colab runtime.
- The earlier environment inspection recorded both packages as unavailable before installation, providing a transparent **pre-installation and post-installation record**.
- The installation process did **not** retrain M1 or M2 and did not require additional Roboflow hosted-training credits.
- The runtime is now prepared for the subsequent reproducibility and analytical verification stages.

#### Reproducibility Evidence

- **Python:** 3.13.15
- **Platform:** Linux x86_64
- **Ultralytics:** 8.4.156
- **Roboflow:** 1.5.0
- The package versions recorded above represent the environment used during execution of this reproducibility notebook.

#### Expected Next Step

- Document the dataset structure and seven-class ontology used by the experiments.
- Reconstruct the M1 and M2 experimental configurations from the preserved project evidence.
- Verify the recorded model results and controlled comparison without representing historical Roboflow results as newly generated notebook outputs.

---

## 5. Dataset and Class Ontology

### 5.1 Purpose

- Document the source dataset used for the M1 and M2 experiments.
- Record the dataset size, task type, preprocessing configuration and seven-class ontology.
- Establish the interpretation boundaries of the available PPE classes before analysing model predictions.
- Preserve the dataset configuration required to understand and reproduce the experimental methodology.

### 5.2 Dataset Configuration

- **Dataset:** Site Construction Safety
- **Source platform:** Roboflow Universe
- **Source images:** 3,858 annotated images
- **Task:** Object Detection
- **Number of classes:** 7
- **Licence:** CC BY 4.0
- **Input resolution:** 640 x 640
- **Resize method:** Fit with black-edge padding
- **Purpose of Fit preprocessing:** Preserve image aspect ratio while avoiding cropping and geometric distortion.

### 5.3 Class Ontology

| Class ID | Class |
|---:|---|
| 1 | Person |
| 2 | Helmet |
| 3 | Non-Helmet |
| 4 | Vest |
| 5 | Gloves |
| 6 | Shoes |
| 7 | bare-arms |

### 5.4 Rationale

- The class ontology determines what the detector can explicitly identify.
- `Non-Helmet` represents an explicit negative safety condition in the dataset.
- Equivalent negative classes such as `Non-Vest`, `Non-Gloves` and `Non-Shoes` are **not** included.
- Therefore, failure to detect `Vest`, `Gloves` or `Shoes` must **not** automatically be interpreted as evidence that the corresponding PPE is absent.
- This distinction is important when evaluating the model as a potential safety-monitoring aid rather than treating every missing positive detection as a compliance violation.

### Expected Outcome

- The dataset used by the experiment is clearly documented.
- All seven detection classes are explicitly recorded.
- The limitations imposed by the class ontology are established before model-performance interpretation.
- Subsequent error analysis can distinguish between a genuine detection failure and a safety condition that the class structure cannot explicitly represent.

---

### 5.5 Dataset Version Structure

#### Purpose

- Record the dataset versions created during the experimental workflow.
- Distinguish the original M1 baseline from the intermediate V2 configuration and the final M2 targeted-augmentation experiment.
- Demonstrate that the M1 and M2 comparison retained the same absolute validation and test sets.

#### Dataset Versions

| Version | Experimental Purpose | Total Images | Training | Validation | Test | Training Status |
|---|---|---:|---:|---:|---:|---|
| V1 / M1 | Baseline dataset | 3,858 | 3,287 | 340 | 231 | Trained and evaluated |
| V2 | Higher-augmentation experimental configuration | 10,432 | 9,861 | 340 | 231 | Generated but not trained |
| V3 / M2 | Resource-optimised targeted augmentation | 7,145 | 6,574 | 340 | 231 | Trained and evaluated |

#### Rationale

- **V1 / M1** established the baseline using the original 3,858 annotated images without augmentation.
- **V2** explored a higher augmentation multiplier, producing 10,432 images, but was not trained after the substantially greater hosted-training resource requirement was identified.
- **V3 / M2** used a more resource-efficient 2x targeted-augmentation strategy and was subsequently trained and evaluated.
- The **validation set remained at 340 images** and the **test set remained at 231 images** between M1 and M2.
- The increased M2 dataset size therefore resulted from augmentation of the training data rather than expansion of the held-out evaluation sets.
- Retaining the same absolute validation and test sets supports a more controlled comparison of M1 and M2.

#### Expected Outcome

- The progression from baseline to targeted augmentation is transparently documented.
- V2 is retained as a legitimate experimental configuration rather than being incorrectly represented as a trained model.
- The controlled relationship between M1 and M2 is clearly established before their model configurations and performance results are compared.

---


## 6. M1 - Baseline Experimental Configuration

### 6.1 Purpose

- Establish the baseline model configuration against which the subsequent M2 experiment can be compared.
- Record the model architecture, dataset version, preprocessing and augmentation settings used for the baseline experiment.
- Preserve sufficient configuration information to make the experimental design transparent and auditable.

### 6.2 Baseline Configuration

| Parameter | M1 Configuration |
|---|---|
| Experiment | M1 Baseline |
| Dataset Version | V1 |
| Source Images | 3,858 |
| Training Images | 3,287 |
| Validation Images | 340 |
| Test Images | 231 |
| Model Architecture | YOLO11 Object Detection |
| Model Size | Nano |
| Input Resolution | 640 x 640 |
| Resize Method | Fit with black-edge padding |
| Data Augmentation | None |
| Training Method | Roboflow hosted training |
| Trained Model | `site-construction-safety-1uiqh-1-yolov11n-t1` |

### 6.3 Rationale

- M1 was intentionally configured as a comparatively simple **baseline experiment**.
- No data augmentation was applied, allowing subsequent changes introduced in M2 to be evaluated against a clearly defined starting point.
- The **Nano** model was selected as a computationally efficient YOLO11 architecture suitable for establishing baseline object-detection performance.
- Fit resizing to **640 x 640** preserved image aspect ratio through padding rather than cropping or geometric distortion.
- The baseline therefore provides a reference point for evaluating whether targeted augmentation changes model behaviour or performance.

### Expected Outcome

- A clearly documented baseline configuration is established.
- M1 can be distinguished from the later targeted-augmentation experiment.
- Subsequent M1 performance results can be interpreted in the context of the exact baseline configuration.
- The configuration provides an auditable reference for the controlled M1 versus M2 comparison.

---

## 7. M1 - Performance Results

### 7.1 Purpose

- Record the principal held-out performance results obtained from the M1 baseline experiment.

- Document both headline metrics and class-level test performance.

- Establish the quantitative baseline for the later controlled comparison with M2.

- Identify class-level weaknesses that informed subsequent external challenge testing and error analysis.

### 7.2 Headline Performance

| Metric | M1 Result |
|---|---:|
| mAP@50 | 96.93% |
| mAP@50:95 | 80.63% |
| Precision | 97.1% |
| Recall | 94.2% |
| F1 | 95.6% |

### 7.3 Held-Out Test Performance by Class

| Class | M1 mAP@50 |
|---|---:|
| All | 97% |
| Gloves | 98% |
| Helmet | 99% |
| Non-Helmet | 97% |
| Person | 99% |
| Shoes | 97% |
| Vest | 99% |
| bare-arms | 91% |

### 7.4 Rationale and Interpretation

- M1 achieved strong overall performance on the held-out evaluation data, with **96.93% mAP@50**, **80.63% mAP@50:95**, **97.1% precision**, **94.2% recall** and **95.6% F1**.

- Class-level test performance was high across all seven detection classes.

- `Person`, `Helmet` and `Vest` achieved particularly strong held-out test results.

- `bare-arms` recorded the lowest class-level test mAP@50 at **91%**.

- The weaker `bare-arms` result provided a specific class-level target for subsequent error analysis.

- Strong held-out performance alone does **not** establish equivalent performance on unfamiliar construction-site imagery.

- External challenge testing was therefore used to investigate generalisation beyond the source dataset.

### Expected Outcome

- The M1 baseline performance is preserved as an auditable experimental record.

- Both overall and class-level results are available for comparison with M2.

- The weakest M1 test class is explicitly identified rather than hidden by the strong overall metric.

- The quantitative results provide the evidence base for the subsequent external challenge-testing stage.

---



## 8. External Challenge Testing and Error Analysis

### 8.1 Purpose

- Evaluate the M1 baseline on images outside the original source dataset.
- Investigate whether strong held-out performance generalises to unfamiliar real-world imagery.
- Identify practical failure modes that may not be apparent from aggregate mAP, precision and recall.
- Use observed failures to inform the targeted augmentation strategy for M2.

### 8.2 Evaluation Protocol

- External images were selected from sources outside the 3,858-image project dataset.
- Images were tested without intentionally modifying their original framing for the challenge evaluation.
- **Confidence threshold:** 50%.
- **Overlap threshold:** 50%.
- The same thresholds were maintained across the external M1 tests to support consistent interpretation.
- Test images included challenging conditions such as partial occlusion, multiple workers, exposed arms, missing helmets, close-up PPE, small/background workers and unusual framing.
- These tests are treated as **qualitative external generalisation tests**, not as a formal external mAP benchmark, because the images were not prepared as a fully annotated evaluation dataset.

### 8.3 Observed M1 Failure Modes

| Error ID | Image Condition | Expected Detection | Observed Issue | Likely Challenge | Proposed Mitigation |
|---|---|---|---|---|---|
| E01 | Worker without helmet | Non-Helmet | Missed detection | Domain shift | Additional targeted examples |
| E02 | Clearly exposed arms | bare-arms | Missed detection | Appearance variation | Augmentation and additional data |
| E03 | Close-up visible glove | Gloves | Missed detection | Crop/scale variation | Crop augmentation |
| E04 | Small/background worker | Person | Missed detection | Small-object detection | Additional data / higher-resolution investigation |
| E05 | Unusual viewpoint or framing | Person/PPE | Missed detection | Camera-angle and framing variation | Viewpoint augmentation |

### 8.4 Rationale and Interpretation

- External testing revealed weaknesses that were not obvious from the approximately **97% held-out mAP@50**.
- `Non-Helmet` and `bare-arms` produced safety-relevant false negatives in selected external scenarios.
- A clearly visible close-up glove was missed in one challenge image, indicating sensitivity to framing or scale.
- Some small, background or partially occluded workers were not detected as `Person`.
- In some unusual external scenarios, the model produced no detections above the fixed 50% confidence threshold.
- These observations demonstrate **domain-shift and generalisation risk**, rather than contradicting the held-out test results.
- The external results were used diagnostically to design the M2 augmentation experiment rather than to calculate an unsupported external accuracy or mAP value.

### 8.5 Ontology Constraint During Error Analysis

- `Non-Helmet` is an explicit class, so a clearly visible uncovered head can support analysis of a missed `Non-Helmet` detection.
- The ontology does **not** include `Non-Vest`, `Non-Gloves` or `Non-Shoes`.
- Therefore, absence of `Vest`, `Gloves` or `Shoes` detections cannot automatically be interpreted as detection of a corresponding safety violation.
- A carried or held helmet must also be interpreted cautiously because object presence and correct PPE usage are not necessarily equivalent.
- These ontology constraints were applied when classifying external-test observations to avoid overstating model failures or safety conclusions.

### Expected Outcome

- External testing provides evidence of practical generalisation weaknesses despite strong held-out metrics.
- Specific and defensible M1 failure modes are documented.
- Error analysis provides a traceable rationale for the targeted augmentations introduced in M2.
- Safety-related conclusions remain bounded by the capabilities of the seven-class ontology and the qualitative nature of the external test set.

---

## 9. M2 - Targeted Augmentation Strategy

### 9.1 Purpose

- Define the experimental changes introduced after analysis of the M1 external challenge-test failures.
- Test whether targeted augmentation could improve robustness while preserving strong held-out performance.
- Keep the principal experimental variables controlled so that M1 and M2 remain meaningfully comparable.

### 9.2 Experimental Hypothesis

- **Hypothesis:** Targeted augmentation addressing orientation, framing, scale and lighting variation may improve model robustness while maintaining approximately equivalent performance on the unchanged held-out validation and test sets.
- The experiment does **not** assume that augmentation must improve every metric or every class.
- A change in the balance between precision and recall is considered part of the experimental result rather than automatically being classified as an improvement or failure.

### 9.3 Targeted Augmentations

| Augmentation | M2 Configuration | Experimental Rationale |
|---|---|---|
| Horizontal Flip | Applied | Introduce left-right viewpoint variation |
| Random Crop | 0% to 10% | Increase robustness to partial framing and scale variation |
| Rotation | -15° to +15° | Introduce moderate camera-angle and orientation variation |
| Brightness | -15% to +15% | Introduce moderate lighting variation |
| Augmentation Multiplier | 2x | Increase training variation while controlling resource requirements |

### 9.4 M2 Dataset Configuration

| Parameter | M2 |
|---|---|
| Dataset Version | V3 |
| Total Images | 7,145 |
| Training Images | 6,574 |
| Validation Images | 340 |
| Test Images | 231 |
| Model Architecture | YOLO11 Object Detection |
| Model Size | Nano |
| Input Resolution | 640 x 640 |
| Resize Method | Fit with black-edge padding |
| Initial Checkpoint | COCO |
| Training Method | Roboflow hosted training |
| Trained Model | `site-construction-safety-1uiqh-3-yolov11n-t1` |

### 9.5 Controlled Experimental Design

- M2 retained the same underlying annotated source dataset used for M1.
- The model family and **Nano** model size were retained.
- Input preprocessing remained **640 x 640 using Fit with black-edge padding**.
- The absolute validation set remained at **340 images**.
- The absolute test set remained at **231 images**.
- The principal experimental change was therefore the introduction of targeted augmentation to the training data.
- Keeping the held-out evaluation sets unchanged supports a more controlled M1 versus M2 comparison.

### 9.6 Resource-Optimised Design Decision

- An intermediate V2 configuration used a higher augmentation multiplier and produced **10,432 total images**.
- V2 was generated but was **not trained** after its substantially greater hosted-training resource requirement was identified.
- V3/M2 therefore used a more resource-efficient **2x augmentation strategy**, producing **7,145 total images**.
- V2 remains part of the experimental record and is not represented as a trained model.
- This decision demonstrates that experimental design considered both methodological value and available computational resources.

### Expected Outcome

- The relationship between M1 error analysis and the M2 augmentation strategy is explicit and traceable.
- M2 differs from M1 in a controlled and documented manner.
- The unchanged held-out validation and test sets provide a consistent basis for performance comparison.
- Resource constraints and the decision not to train V2 are transparently documented rather than omitted.
- M2 performance can subsequently be evaluated against the stated experimental hypothesis.

---


## 10. M2 - Performance Results

### 10.1 Purpose

- Record the principal held-out performance results obtained from the M2 targeted-augmentation experiment.

- Compare M2 class-level behaviour with the M1 baseline using the unchanged held-out test set.

- Determine whether targeted augmentation maintained overall performance while changing performance for particular classes.

- Preserve both favourable and unfavourable changes as part of the experimental record.

### 10.2 Headline Performance

| Metric | M2 Result |
|---|---:|
| mAP@50 | 96.69% |
| mAP@50:95 | 78.58% |
| Precision | 95.7% |
| Recall | 94.8% |
| F1 | 95.2% |

### 10.3 Held-Out Test Performance by Class

| Class | M2 mAP@50 |
|---|---:|
| All | 97% |
| Gloves | 97% |
| Helmet | 99% |
| Non-Helmet | 96% |
| Person | 99% |
| Shoes | 97% |
| Vest | 99% |
| bare-arms | 93% |

### 10.4 Rationale and Interpretation

- M2 achieved **96.69% mAP@50**, **78.58% mAP@50:95**, **95.7% precision**, **94.8% recall** and **95.2% F1**.

- Overall mAP@50 remained close to the M1 baseline after targeted augmentation, while mAP@50:95 was lower than the M1 result.

- `Person`, `Helmet` and `Vest` continued to demonstrate particularly strong held-out test performance.

- `bare-arms`, which was the weakest M1 test class, increased from **91% to 93% mAP@50**.

- M2 did not improve every class or every headline metric.

- The results therefore do not support describing M2 as universally superior to M1.

- The appropriate interpretation is that targeted augmentation altered the model's performance characteristics: recall and `bare-arms` performance improved, while precision and mAP@50:95 declined and overall mAP@50 remained broadly stable.

### 10.5 External Validation Boundary

- M1 external challenge testing identified practical generalisation weaknesses and informed the M2 augmentation strategy.

- Equivalent external challenge-image inference evidence was not available for M2 during the experimental period.

- Consequently, no claim is made that M2 improved external-image performance.

- Repeating the same external challenge tests on M2 remains an important future validation step.

### Expected Outcome

- M2 performance is documented independently before direct comparison with M1.

- Class-level changes are visible rather than being obscured by the overall mAP result.

- The improvement in `bare-arms` is recorded without implying universal model improvement.

- The reduction in mAP@50:95 is explicitly preserved as part of the experimental record.

- Unsupported claims about M2 external generalisation are explicitly avoided.

- The results provide the quantitative basis for the controlled M1 versus M2 comparison in the next section.

---

## 11. Controlled M1 vs M2 Comparison

### 11.1 Purpose

- Reproduce the direct quantitative comparison between the recorded M1 and M2 experimental results.
- Calculate performance changes programmatically rather than relying only on manually prepared comparison tables.
- Examine both headline metrics and class-level held-out test performance.
- Provide reproducible numerical evidence showing where M2 improved, declined or remained unchanged relative to M1.

### 11.2 Rationale

- M1 and M2 used the same model family, model size, input resolution and absolute held-out validation and test sets.
- The principal experimental difference was the introduction of targeted training-data augmentation in M2.
- This makes direct comparison of the recorded held-out results meaningful within the boundaries of the experiment.
- Recalculating the differences in Python provides an independently executable verification of the reported percentage-point changes.
- The code does **not** regenerate the original Roboflow metrics; it operates on the preserved experimental results recorded in Sections 7 and 10.
- This distinction prevents historical hosted-training outputs from being misrepresented as newly generated model-evaluation results.

### Expected Outcome

- Construct structured M1 and M2 result tables from the preserved experimental values.
- Calculate M2 minus M1 changes automatically.
- Verify the reported headline and class-level differences.
- Produce reproducible comparison outputs suitable for subsequent visualisation and interpretation.

---

In [13]:
# ==============================================================
# MAICEN-0526 GROUP 5 - CONTROLLED M1 VS M2 COMPARISON
# ==============================================================

import pandas as pd

# --------------------------------------------------------------
# Headline metrics
# Values are preserved historical results from Roboflow.
# This code verifies the comparison; it does not regenerate them.
# --------------------------------------------------------------

headline_results = pd.DataFrame({
    "Metric": ["mAP@50", "mAP@50:95", "Precision", "Recall", "F1"],
    "M1 (%)": [96.93, 80.63, 97.1, 94.2, 95.6],
    "M2 (%)": [96.69, 78.58, 95.7, 94.8, 95.2]
})

headline_results["Change (pp)"] = (
    headline_results["M2 (%)"] - headline_results["M1 (%)"]
).round(2)

# --------------------------------------------------------------
# Held-out test performance by class
# --------------------------------------------------------------

class_results = pd.DataFrame({
    "Class": [
        "All",
        "Gloves",
        "Helmet",
        "Non-Helmet",
        "Person",
        "Shoes",
        "Vest",
        "bare-arms"
    ],
    "M1 mAP@50 (%)": [97, 98, 99, 97, 99, 97, 99, 91],
    "M2 mAP@50 (%)": [97, 97, 99, 96, 99, 97, 99, 93]
})

class_results["Change (pp)"] = (
    class_results["M2 mAP@50 (%)"] -
    class_results["M1 mAP@50 (%)"]
)

# --------------------------------------------------------------
# Display reproducible comparison
# --------------------------------------------------------------

print("=" * 72)
print("MAICEN-0526 GROUP 5 - CONTROLLED M1 VS M2 COMPARISON")
print("=" * 72)

print("\nHEADLINE PERFORMANCE")
print("-" * 72)
print(headline_results.to_string(index=False))

print("\nHELD-OUT TEST PERFORMANCE BY CLASS")
print("-" * 72)
print(class_results.to_string(index=False))

print("\n" + "=" * 72)
print("Comparison calculation complete.")
print("=" * 72)

MAICEN-0526 GROUP 5 - CONTROLLED M1 VS M2 COMPARISON

HEADLINE PERFORMANCE
------------------------------------------------------------------------
   Metric  M1 (%)  M2 (%)  Change (pp)
   mAP@50   96.93   96.69        -0.24
mAP@50:95   80.63   78.58        -2.05
Precision   97.10   95.70        -1.40
   Recall   94.20   94.80         0.60
       F1   95.60   95.20        -0.40

HELD-OUT TEST PERFORMANCE BY CLASS
------------------------------------------------------------------------
     Class  M1 mAP@50 (%)  M2 mAP@50 (%)  Change (pp)
       All             97             97            0
    Gloves             98             97           -1
    Helmet             99             99            0
Non-Helmet             97             96           -1
    Person             99             99            0
     Shoes             97             97            0
      Vest             99             99            0
 bare-arms             91             93            2

Comparison calculation

### 11.4 Controlled Comparison Interpretation

- The programmatic comparison confirms that M2 retained very similar overall mAP@50 performance to M1, with **mAP@50 changing from 96.93% to 96.69% (-0.24 percentage points)**.

- At the stricter IoU range, **mAP@50:95 decreased from 80.63% to 78.58% (-2.05 pp)**, showing that the effect of M2 is less favourable when detection performance is evaluated across multiple IoU thresholds.

- **Recall increased from 94.2% to 94.8% (+0.6 pp)**, while **precision decreased from 97.1% to 95.7% (-1.4 pp)** and **F1 decreased from 95.6% to 95.2% (-0.4 pp)**.

- At class level, the principal improvement occurred for `bare-arms`, the weakest M1 test class, which increased from **91% to 93% mAP@50 (+2 pp)**.

- `Gloves` decreased from **98% to 97% (-1 pp)** and `Non-Helmet` decreased from **97% to 96% (-1 pp)**. `Helmet`, `Person`, `Shoes` and `Vest` remained unchanged.

- These results indicate a **trade-off rather than a universal improvement**: targeted augmentation improved the weakest M1 class and slightly increased overall recall, while mAP@50 remained broadly stable, but precision, F1 and mAP@50:95 declined.

- Because M1 and M2 retained the same absolute validation and test sets, these differences provide a controlled indication of the effect of the M2 targeted-augmentation strategy within the held-out dataset.

- The results do **not** establish that M2 generalises better to unfamiliar construction-site imagery. Equivalent external challenge testing of M2 would be required before making that conclusion.

### 11.5 Experimental Finding

The M2 experiment demonstrates that targeted augmentation can alter model behaviour without producing a universal performance improvement. The improvement in `bare-arms` is relevant because this class had been identified as the weakest M1 held-out class and as a practical weakness during external challenge testing. However, the simultaneous reductions in mAP@50:95, precision, F1, `Gloves` and `Non-Helmet` show why model refinement should be evaluated across multiple metrics and classes rather than judged from a single aggregate measure.

### Expected Outcome

- The M1 and M2 comparison is supported by reproducibly calculated numerical differences.
- Improvements, declines and unchanged results are reported transparently.
- Both **mAP@50 and mAP@50:95** are incorporated into the experimental interpretation.
- The M2 experiment is interpreted as a measured **performance trade-off**, not as an unsupported claim of universal improvement.
- The distinction between held-out dataset performance and external generalisation is preserved.


## 12. Reproducibility Verification

### 12.1 Purpose

- Verify that the principal experimental configuration, recorded results and M1 versus M2 comparison can be reconstructed from the information preserved in this notebook.
- Confirm that the notebook contains sufficient technical detail for another user to understand and audit the experimental workflow.
- Distinguish clearly between **reproducible analytical verification** performed in this notebook and the original **Roboflow-hosted model training and evaluation**.

### 12.2 Reproducibility Verification Criteria

The following components are checked as part of the reproducibility assessment:

| Component | Verification Requirement |
|---|---|
| Computational environment | Python platform and principal package versions are recorded |
| Dataset | Source size, train/validation/test structure and seven-class ontology are documented |
| M1 configuration | Architecture, model size, resolution, preprocessing and augmentation settings are recorded |
| M1 results | Headline and class-level held-out results are preserved |
| External testing | Test conditions, fixed thresholds, observations and interpretation boundaries are documented |
| M2 configuration | Targeted augmentations and unchanged held-out sets are recorded |
| M2 results | Headline and class-level held-out results are preserved |
| M1 vs M2 comparison | Performance differences are recalculated programmatically |
| Experimental limitations | Boundaries between hosted results, notebook verification and external generalisation are explicitly stated |

### 12.3 Verification Principle

Reproducibility does not require this notebook to claim that the original hosted training runs were recreated from scratch. Instead, the notebook preserves the experimental configuration and evidence generated through Roboflow, records the execution environment, and provides executable analytical steps for verifying the principal comparisons derived from those results.

This approach ensures that **historical experimental evidence is not misrepresented as newly generated output**, while still providing a transparent and auditable technical record of the project.

### Expected Outcome

- The experimental workflow can be followed from the M1 baseline through error analysis, M2 refinement and controlled comparison.
- Important configuration choices and numerical results are traceable within the notebook.
- Executable notebook cells independently verify the environment and M1 versus M2 calculations.
- The reproducibility boundary is explicit and technically defensible.


## 13. Limitations and Responsible Use

### 13.1 Model and Dataset Limitations

- The model was trained on a finite construction-site image dataset and may not represent the full diversity of real construction environments, workers, PPE designs, lighting conditions, camera positions, weather conditions and site types.
- Strong held-out performance does **not** guarantee equivalent performance on unfamiliar real-world imagery.
- M1 external challenge testing demonstrated practical domain-shift effects, including missed detections involving `Non-Helmet`, `bare-arms`, close-up PPE and small, background or partially occluded workers.
- The external challenge images were not a formally annotated benchmark dataset. These tests therefore provide **qualitative evidence of generalisation behaviour**, not an external mAP or accuracy measurement.
- Equivalent external challenge testing was not completed for M2 during the experimental period. No claim is therefore made that M2 improved external-image generalisation.

### 13.2 Class-Ontology Limitation

The seven-class ontology contains:

`Person`, `Helmet`, `Non-Helmet`, `Vest`, `Gloves`, `Shoes` and `bare-arms`.

Only `Non-Helmet` explicitly represents the absence of a PPE item. The ontology does **not** contain corresponding `Non-Vest`, `Non-Gloves` or `Non-Shoes` classes.

Consequently:

- Failure to detect `Vest`, `Gloves` or `Shoes` must **not** automatically be interpreted as evidence that the PPE item is absent.
- A missing positive detection is not equivalent to a detected safety violation.
- A carried or held helmet should not automatically be interpreted as correct helmet usage.
- A practical compliance system would require additional worker-to-PPE association and decision logic beyond individual object detections.

### 13.3 Safety Risk and Human Oversight

In a safety-monitoring context, **false negatives are particularly important** because a missed worker or missed safety condition may prevent a potential hazard from being flagged.

However, automated detections should not independently determine whether a worker or site is compliant. Model outputs should instead be treated as decision-support information requiring appropriate human review.

A practical deployment architecture could therefore separate:

**Object Detection → Worker/PPE Association → Compliance Rules → Human Review / Alert**

This would prevent raw detector output from being treated as an authoritative safety judgement.

### 13.4 Bias and Generalisation Considerations

Potential performance variation may arise from differences in:

- construction-site environments and geographic contexts;
- worker appearance, posture and degree of occlusion;
- PPE colour, style, shape and manufacturer;
- camera distance, resolution, viewpoint and orientation;
- lighting, shadows and weather;
- object scale and background complexity.

Performance should therefore be validated on representative deployment data before operational use.

### 13.5 Responsible-Use Position

This project is an **experimental computer-vision safety-monitoring system** developed for educational and analytical purposes.

The model should:

- support rather than replace human safety judgement;
- not be used as the sole basis for disciplinary, employment or compliance decisions;
- communicate confidence and uncertainty where practicable;
- be periodically evaluated for false negatives, false positives and performance drift;
- be validated against representative operational imagery before deployment;
- operate with appropriate privacy, data-governance and human-oversight controls.

### Expected Outcome

- Technical limitations are explicitly documented rather than obscured by high headline metrics.
- The safety implications of false negatives and domain shift are recognised.
- The limitations of the seven-class ontology are clearly separated from model-detection errors.
- Human oversight remains central to any potential operational use.
- The project demonstrates responsible interpretation of computer-vision results rather than treating model output as an authoritative safety determination.


## 14. Conclusion and Recommended Future Work

### 14.1 Purpose

- Consolidate the principal findings from the M1 baseline and M2 targeted-augmentation experiments.
- State what the experimental evidence supports without overstating model capability.
- Identify the most important technical limitations revealed by the project.
- Define evidence-based priorities for future model development and validation.

### 14.2 Principal Conclusions

This project developed and evaluated a seven-class construction-site PPE object-detection system using a controlled **M1 baseline → external challenge testing → error analysis → M2 targeted augmentation → M1 versus M2 comparison** workflow.

The principal findings are:

- **M1 achieved strong held-out performance**, recording **96.9% mAP@50, 97.1% precision, 94.2% recall and 95.6% F1**.
- Despite these strong held-out results, external challenge testing identified practical weaknesses involving **Non-Helmet, bare-arms, close-up or cropped PPE, and small, background or partially occluded workers**.
- These observations demonstrated that strong performance on held-out source-dataset images does **not** by itself establish equivalent performance on unfamiliar construction-site imagery.
- M2 introduced controlled targeted augmentation while retaining the same model family, model size, input resolution and absolute validation and test sets.
- M2 recorded **96.7% mAP@50, 95.7% precision, 94.8% recall and 95.2% F1**.
- The weakest M1 held-out class, **bare-arms**, improved from **91% to 93% mAP@50 (+2 pp)**, while overall recall increased by **0.6 pp**.
- At the same time, overall precision decreased by **1.4 pp**, while **Gloves** and **Non-Helmet** each decreased by **1 pp** on the held-out test set.
- M2 therefore represents a **measured performance trade-off rather than a universal improvement**.
- Equivalent external challenge-image inference was not available for M2 during the experimental period; consequently, the project does **not** claim that M2 improved external generalisation.

### 14.3 Recommended Future Work

Future development should prioritise:

1. **Repeat the M1 external challenge tests using M2** under the same fixed inference conditions to directly evaluate whether targeted augmentation improved practical generalisation.
2. **Expand representative training data** for difficult conditions including bare arms, uncovered heads, close-up PPE, partial occlusion, small or distant workers, unusual viewpoints and varied lighting.
3. **Review the class ontology**, particularly the absence of explicit `Non-Vest`, `Non-Gloves` and `Non-Shoes` classes, before interpreting missing positive PPE detections as safety violations.
4. **Investigate worker-to-PPE association logic** so detected PPE can be linked to the appropriate individual rather than interpreted only as independent objects within an image.
5. **Evaluate additional model configurations and input resolutions** where computational resources permit, particularly for small-object detection.
6. **Create a formally annotated external benchmark** containing representative construction-site imagery so future models can be evaluated quantitatively beyond the source dataset.
7. **Maintain human oversight** and validate the system on representative operational imagery before considering any real-world safety-monitoring deployment.

### 14.4 Final Experimental Position

The project demonstrates that model evaluation should extend beyond a single headline accuracy metric. The combination of held-out testing, external challenge testing, class-level error analysis, controlled model refinement, reproducible comparison and explicit governance boundaries provides a more defensible assessment of computer-vision performance.

The resulting models should therefore be regarded as **experimental PPE-detection systems and potential decision-support tools**, not as autonomous safety-compliance authorities.

### Expected Outcome

- The experimental findings are consolidated without overstating model performance.
- The distinction between held-out performance and external generalisation remains explicit.
- Future work is directly connected to weaknesses observed during the experiments.
- The notebook concludes with a technically defensible and responsible statement of model capability.

## 15. Reproducibility Checklist and Completion Status

### 15.1 Reproducibility Checklist

The following checklist summarises the evidence preserved within this notebook for independent review of the experimental workflow.

| Reproducibility Component | Status | Evidence Preserved |
|---|---|---|
| Computational environment | Complete | Python version, operating system and principal package versions recorded |
| Project dependencies | Complete | Ultralytics and Roboflow installation and post-installation verification recorded |
| Dataset provenance | Complete | Source dataset, licence, image count and experimental use documented |
| Dataset structure | Complete | Training, validation and test image counts recorded |
| Class ontology | Complete | Seven detection classes and ontology limitations documented |
| M1 baseline configuration | Complete | Architecture, model size, resolution, preprocessing and augmentation settings recorded |
| M1 held-out results | Complete | Headline metrics and class-level test mAP@50 preserved |
| M1 external challenge testing | Complete | Fixed inference conditions, observations, failure modes and interpretation boundaries documented |
| Error analysis | Complete | Principal false-negative and domain-shift observations documented |
| M2 experimental rationale | Complete | Targeted augmentations linked to M1 error analysis |
| M2 configuration | Complete | Dataset size, augmentation strategy, architecture and unchanged held-out sets recorded |
| M2 held-out results | Complete | Headline metrics and class-level test mAP@50 preserved |
| M1 versus M2 comparison | Complete | Percentage-point differences recalculated programmatically |
| Governance and limitations | Complete | Safety, ontology, generalisation, privacy and human-oversight boundaries documented |
| M2 external challenge testing | Future validation | Equivalent external challenge-image inference was not available during the experimental period |

### 15.2 Reproducibility Boundary

This notebook provides a reproducible and auditable record of the experimental design, computational environment, preserved Roboflow results and analytical comparison performed in this project.

The original M1 and M2 hosted training runs are **not rerun inside this notebook**. Their historical metrics are preserved as experimental evidence and are clearly distinguished from calculations executed directly within the notebook.

Accordingly, reproducibility in this submission means that an independent reviewer can:

1. identify the dataset structure and class ontology;
2. reconstruct the documented M1 and M2 experimental configurations;
3. inspect the preserved model-performance evidence;
4. reproduce the numerical M1 versus M2 comparison from the recorded results;
5. trace the progression from baseline testing through error analysis and targeted refinement; and
6. understand the limitations governing interpretation and future deployment.

### 15.3 Completion Statement

The reproducibility notebook is complete when read together with the supporting project repository and preserved experimental evidence.

It provides a transparent technical record of the project from **baseline model development through controlled refinement, quantitative comparison, error analysis, governance and recommended future validation**.

No historical training result is presented as newly generated notebook output, and no claim of improved external generalisation is made without equivalent M2 external testing.

**Notebook status: COMPLETE**
